# [1.2] Code-Along: Reading a Transformer's Mind

You've already built a transformer from scratch, so you know *what the computation is*. This code-along is about the next question: **given a trained model, how do we figure out *what algorithm it learned*?**

We'll go from "poke at activations" to "read the algorithm straight out of the weight matrices", and by the end you'll have reverse-engineered a real circuit, the **induction circuit**, three separate ways (attention patterns, causal interventions, and pure weight analysis).

**How to use this notebook**

- Read the 💡 **Intuition** blocks. They're the point of the tutorial; the code just checks them.
- Every ✏️ **Code-along** cell has a stub. Fill it in, run it, then run the ✅ check cell below it.
- Stuck for more than ~10 minutes? Open the collapsed *Solution* block under the stub. There's also a fully filled-in twin: `1.2_code_along_solutions.ipynb`.
- It follows the structure of the ARENA notebook `1.2_Intro_to_Mech_Interp_exercises.ipynb`, but trimmed to the core ideas with more "why". Use the ARENA notebook afterwards for bonus material.

| Part | Idea | Tool you'll build |
|---|---|---|
| 0 | The residual stream is a sum | (mental model) |
| 1 | A transformer with a microscope attached | recompute attention from cached q, k |
| 2 | Heads have "jobs" you can see | attention-pattern head detectors |
| 3 | Induction: the algorithm behind in-context copying | repeated-token loss test, induction detector |
| 4 | Hooks: measure at scale | batched induction scores (toy model → GPT-2) |
| 5 | Who wrote the answer? | direct logit attribution |
| 6 | What breaks if I remove it? | ablation |
| 7 | Read the algorithm from the weights | OV / QK / K-composition circuits |

## 0️⃣ The one mental model you need

When you built the transformer, each block did `resid = resid + attn(resid)` then `resid = resid + mlp(resid)`. Flip your perspective on that:

> **The residual stream is a shared scratchpad (a "bus"). Every component *reads* from it with a linear map and *writes* to it by *adding* a vector.**

```
 embed ──┬──────────────┬──────────────┬──────────► resid_final ──► W_U ──► logits
         │              │              │
         ▼  read        ▼  read        ▼  read
       head 0.0 ...   head 1.0 ...   ...
         │  write (+)   │  write (+)
         └──────────────┴── all added into the same stream
```

Three consequences drive *everything* in this notebook:

1. **The final residual is a sum.** `resid_final = embed + Σ_heads head_output (+ MLPs)`. If there is no nonlinearity after it (our toy model has no LayerNorm), then `logits = resid_final @ W_U` is **also** a sum, one term per component. So we can ask *"how much did head 1.4 contribute to the correct answer?"* → **Part 5, direct logit attribution.**
2. **Each attention head is two independent circuits.**
   - **QK circuit** `W_Q W_Kᵀ` (d_model × d_model): decides *where to look*, i.e. the attention pattern.
   - **OV circuit** `W_V W_O` (d_model × d_model): decides *what gets moved* once you've looked.

   You implemented these as four matrices; the insight is that only these two *products* matter. → **Part 7.**
3. **Later heads can read what earlier heads wrote.** A layer-1 head's query, key, or value input can include a layer-0 head's output. This is **composition** (Q-, K-, V-composition), and it's how 2-layer models do things 1-layer models can't. → **Parts 3 & 7.**

## Setup

In [1]:
import functools
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import plotly.express as px
import torch as t
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, FactoredMatrix, HookedTransformer, HookedTransformerConfig, utils
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

# We're only doing inference / analysis, never training. Turning grads off saves memory and time.
t.set_grad_enabled(False)


def heatmap(x, title="", xlabel="Head", ylabel="Layer", **kwargs):
    '''Small helper so plotting never distracts from the ideas.'''
    fig = px.imshow(
        utils.to_numpy(x), title=title, labels={"x": xlabel, "y": ylabel},
        color_continuous_scale="RdBu", color_continuous_midpoint=0.0, **kwargs,
    )
    fig.show()

device: mps


---
## 1️⃣ A transformer with a microscope attached

`HookedTransformer` (from TransformerLens) is the model you already wrote, plus one trick: at every interesting activation there's a `HookPoint`, an identity module (`x -> x`) that we can **read** or **overwrite** from the outside.

Let's load GPT-2 small (85M params, 12 layers × 12 heads).

In [2]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small", device=str(device))

cfg = gpt2_small.cfg
print(f"layers={cfg.n_layers} heads={cfg.n_heads} d_model={cfg.d_model} d_head={cfg.d_head} d_vocab={cfg.d_vocab}")

/Users/macbookm1/Documents/ARENA_3.0/.venv/lib/python3.11/site-packages/transformer_lens/HookedTransformerConfig.py:343: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.14.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  utils.warn_if_mps(self.device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
layers=12 heads=12 d_model=768 d_head=64 d_vocab=50257


### Tokens, the 5-minute version

`model.to_tokens(text)` → `[batch, seq]` int tensor, **with a BOS token prepended** (`<|endoftext|>` for GPT-2). The BOS token is not decoration: many heads use it as a "nowhere to look" resting spot (you'll see this in Part 2).

`model.to_str_tokens(text)` → list of strings, handy for labelling plots.

In [3]:
print(gpt2_small.to_str_tokens("Mechanistic interpretability"))
print(gpt2_small.to_tokens("Mechanistic interpretability"))

['<|endoftext|>', 'Mech', 'an', 'istic', ' interpret', 'ability']
tensor([[50256, 28452,   272,  2569,  6179,  1799]], device='mps:0')


### ✏️ Code-along 1.1: How often is GPT-2 right?

Warm-up to get comfortable with the shapes. Run the model on `text`, take the argmax prediction at every position, and compare it to the *actual next token*.

💡 **Intuition:** the logits at position `i` are a prediction for token `i+1`. So predictions `[:-1]` line up with tokens `[1:]`. Getting this off-by-one right is 80% of the bugs in interp code.

In [4]:
text = "The Eiffel Tower is located in the city of Paris. The Eiffel Tower is located in the city of Paris."

logits: Tensor = gpt2_small(text, return_type="logits")  # [batch=1, seq, d_vocab]
tokens = gpt2_small.to_tokens(text)                         # [batch=1, seq]

prediction = logits.argmax(dim=-1).squeeze()[:-1]  # [seq-1]
true_next = tokens.squeeze()[1:]                     # [seq-1]
is_correct = prediction == true_next

print(f"Accuracy: {is_correct.sum()}/{len(true_next)}")
print("Correctly predicted:", gpt2_small.to_str_tokens(prediction[is_correct]))

Accuracy: 17/26
Correctly predicted: ['el', ' Tower', ' is', ' in', ' the', ' of', 'iff', 'el', ' Tower', ' is', ' located', ' in', ' the', ' city', ' of', ' Paris', '.']


👀 **Notice:** the second sentence is predicted *much* better than the first. Nothing about Paris got smarter; the model is **copying from earlier in the context**. Hold on to that observation: explaining it is the whole of Part 3.

### Caching every activation

`run_with_cache` runs the model once and stores *every* hook point's value in an `ActivationCache` (a dict).

Naming convention (you'll use these constantly):

| Full name | Shorthand | Shape (batch removed) | What it is |
|---|---|---|---|
| `hook_embed` | `"embed"` | `[seq, d_model]` | token embeddings |
| `hook_pos_embed` | `"pos_embed"` | `[seq, d_model]` | positional embeddings |
| `blocks.{L}.hook_resid_pre` | `"resid_pre", L` | `[seq, d_model]` | residual stream entering block L |
| `blocks.{L}.attn.hook_q` / `k` / `v` | `"q", L` | `[seq, n_heads, d_head]` | queries / keys / values |
| `blocks.{L}.attn.hook_pattern` | `"pattern", L` | `[n_heads, seq_q, seq_k]` | attention probabilities |
| `blocks.{L}.attn.hook_z` | `"z", L` | `[seq, n_heads, d_head]` | pattern-weighted values (before `W_O`) |
| `blocks.{L}.attn.hook_result` | `"result", L` | `[seq, n_heads, d_model]` | each head's output, *after* `W_O` (only if `cfg.use_attn_result=True`) |

`utils.get_act_name("pattern", 0)` converts shorthand → full name.

In [5]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

t.testing.assert_close(gpt2_cache["pattern", 0], gpt2_cache["blocks.0.attn.hook_pattern"])
print(gpt2_cache["pattern", 0].shape, "  <- [n_heads, seq_q, seq_k]")

torch.Size([12, 33, 33])   <- [n_heads, seq_q, seq_k]


### ✏️ Code-along 1.2: Rebuild attention from the cache

You've written attention before. Prove to yourself the microscope shows exactly that computation: take the cached `q` and `k` for layer 0 and rebuild the attention pattern.

Steps: dot product (`q·k` per head) → causal mask (query can't see future keys) → scale by `1/√d_head` → softmax over keys.

In [6]:
q, k = gpt2_cache["q", 0], gpt2_cache["k", 0]   # each [seq, n_heads, d_head]
seq, n_heads, d_head = q.shape

scores = einops.einsum(q, k, "seqQ n h, seqK n h -> n seqQ seqK")
mask = t.triu(t.ones((seq, seq), dtype=t.bool, device=q.device), diagonal=1)  # True above diagonal = future
scores = scores.masked_fill(mask, float("-inf"))
layer0_pattern_from_q_and_k = (scores / d_head**0.5).softmax(dim=-1)

In [7]:
# ✅ check
t.testing.assert_close(gpt2_cache["pattern", 0], layer0_pattern_from_q_and_k)
print("Matches the model's own attention pattern!")

Matches the model's own attention pattern!


### Looking at attention

`circuitsvis` renders patterns interactively. Hover a token to see where it attends (**dest = query**, the token doing the looking; **src = key**, the token being looked at).

In [8]:
display(cv.attention.attention_patterns(
    tokens=gpt2_small.to_str_tokens(gpt2_text),
    attention=gpt2_cache["pattern", 0],
))

---
## 2️⃣ Heads have jobs you can *see*

GPT-2 has MLPs and LayerNorms, which make exact analysis messy. So we switch to a **model organism**: a 2-layer, **attention-only** transformer, no LayerNorm, trained on real text.

💡 **Why this model is perfect for learning:**
- No MLPs and no LayerNorm ⇒ everything except the attention softmax is **linear**. That's what lets us read the algorithm out of the weights in Part 7.
- 2 layers is the *minimum* depth for composition, and it's exactly enough to learn induction.
- **Shortformer positional embeddings:** position info is added *only* to the inputs of Q and K, **not** into the residual stream. So the residual stream carries *token* information only; position only affects *where heads look*. Remember this for Part 5.

In [9]:
toy_cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,          # store per-head outputs (hook_result), we need these in Part 5
    normalization_type=None,       # no LayerNorm
    positional_embedding_type="shortformer",
    device=str(device),
)

from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(repo_id="callummcdougall/attn_only_2L_half", filename="attn_only_2L_half.pth")
model = HookedTransformer(toy_cfg)
model.load_state_dict(t.load(weights_path, map_location=device, weights_only=True))
print("Toy model loaded.")

Toy model loaded.


In [10]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)
str_tokens = model.to_str_tokens(text)

for layer in range(model.cfg.n_layers):
    print(f"Layer {layer}")
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", layer]))

Layer 0


Layer 1


Click through the heads above. You'll see three recurring "shapes" of attention pattern:

| Pattern | What it looks like | Plausible job |
|---|---|---|
| **current-token** | bright main diagonal | "look at myself" (re-process my own token) |
| **previous-token** | bright stripe *just below* the diagonal | "copy what the previous token was into my position" |
| **first-token (BOS)** | bright first column | a *no-op*: park attention on BOS when nothing is relevant |

💡 **Why BOS heads exist:** softmax forces attention to sum to 1. A head that has "nothing to say" at this position needs somewhere harmless to dump its attention; BOS is the conventional sink.

### ✏️ Code-along 2.1: Write head detectors

Turn eyeballing into code. Write `detect_heads(cache, score_fn)`, which scores every head with `score_fn(pattern) -> float` and returns names like `"0.7"` for heads scoring above `threshold`. Then write the three score functions.

Hint: `pattern.diagonal(offset)` gives you diagonals. `offset=-1` is the stripe one below the main diagonal.

In [11]:
def detect_heads(cache: ActivationCache, score_fn: Callable[[Tensor], float], threshold: float = 0.4) -> list[str]:
    '''Returns ["layer.head", ...] for every head whose score_fn(pattern[seq_q, seq_k]) > threshold.'''
    heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            if score_fn(cache["pattern", layer][head]) > threshold:
                heads.append(f"{layer}.{head}")
    return heads


def current_token_score(pattern: Float[Tensor, "seq_q seq_k"]) -> float:
    return pattern.diagonal().mean().item()


def prev_token_score(pattern: Float[Tensor, "seq_q seq_k"]) -> float:
    return pattern.diagonal(-1).mean().item()


def first_token_score(pattern: Float[Tensor, "seq_q seq_k"]) -> float:
    return pattern[:, 0].mean().item()

In [12]:
# ✅ check (expected roughly: current = 0.9, prev = 0.7, first = 0.3 1.4 1.10)
print("current-token heads :", detect_heads(cache, current_token_score))
print("previous-token heads:", detect_heads(cache, prev_token_score))
print("first-token heads   :", detect_heads(cache, first_token_score))

current-token heads : ['0.9']
previous-token heads: ['0.7']
first-token heads   : ['0.3', '1.4', '1.10']


**Remember head `0.7` (previous-token head) and heads `1.4` / `1.10`.** On normal text, 1.4 and 1.10 mostly sit on BOS doing nothing. The next section shows the input where they wake up.

---
## 3️⃣ Induction: the algorithm behind in-context copying

Back in 1.1, GPT-2 got the *second* "Eiffel Tower … Paris" almost perfectly. The cleanest way to isolate that ability is a test that *nothing else* can pass:

> Feed the model **random tokens**, then the **same random tokens again**.

- **First half:** random ⇒ unpredictable ⇒ high loss. It's usually even *worse* than uniform guessing (`ln(d_vocab)` ≈ 10.8 nats), because the model confidently predicts plausible English, and random tokens aren't.
- **Second half:** a model that can say *"I've seen this sequence before; last time, what came after the current token?"* gets it nearly right.

💡 **The induction algorithm.** Pattern: `… [A] [B] … [A] → predict [B]`

```
position:    …   j-1   j    …   i  (now)
token:       …    A    B    …   A        ⇒ predict B

Step 1 (layer 0, previous-token head 0.7):
    at every position, write "the token before me was ___" into the residual stream.
    So position j now carries the info "previous token = A".

Step 2 (layer 1, induction head 1.4 / 1.10):
    QUERY at i asks:  "who has previous-token = A?"   (my current token is A)
    KEY   at j says:  "me! my previous token was A"   (written there by 0.7)
    → attend to j, and the OV circuit COPIES token B into the output ⇒ logits for B go up.
```

Two key points:
- The induction head's **key** is built from a *previous head's output*. That's **K-composition**, and it's why you need ≥2 layers.
- On a repeated sequence of length `n`, the token at position `i` in the second half attends to position `i - (n - 1)`, i.e. one *after* its earlier copy. That's a diagonal stripe at offset `-(n-1)`.

### ✏️ Code-along 3.1: The repeated-tokens test

1. `generate_repeated_tokens`: `[BOS, r_1 … r_n, r_1 … r_n]` with random `r_i`, shape `[batch, 1 + 2n]`.
2. `get_log_probs`: for each position, the log-prob the model assigned to the **actual next token**. Shape `[batch, seq-1]`.

Hint for 2: `log_softmax`, then `.gather(-1, index)` with the next tokens as index. Mind the off-by-one from 1.1!

In [13]:
def generate_repeated_tokens(model: HookedTransformer, seq_len: int, batch_size: int = 1) -> Int[Tensor, "batch full_seq"]:
    '''[BOS, random seq_len tokens, same seq_len tokens again]'''
    t.manual_seed(0)  # reproducibility
    bos = t.full((batch_size, 1), model.tokenizer.bos_token_id, dtype=t.long)
    half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.long)
    return t.cat([bos, half, half], dim=-1).to(model.cfg.device)


def get_log_probs(logits: Float[Tensor, "batch seq d_vocab"], tokens: Int[Tensor, "batch seq"]) -> Float[Tensor, "batch seq-1"]:
    '''log_probs[b, s] = log P(tokens[b, s+1] | tokens[b, :s+1])'''
    log_probs = logits.log_softmax(dim=-1)[:, :-1]      # predictions for positions 1..seq-1
    next_tokens = tokens[:, 1:].unsqueeze(-1)            # [batch, seq-1, 1]
    return log_probs.gather(-1, next_tokens).squeeze(-1)

In [14]:
seq_len = 50
rep_tokens = generate_repeated_tokens(model, seq_len, batch_size=1)
rep_logits, rep_cache = model.run_with_cache(rep_tokens)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)

log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

# ✅ check
assert rep_tokens.shape == (1, 2 * seq_len + 1)
assert log_probs.shape == (2 * seq_len,)
print(f"Loss on first half : {-log_probs[:seq_len].mean():.2f} nats")
print(f"Loss on second half: {-log_probs[seq_len:].mean():.2f} nats   <- should be dramatically lower")

px.line(
    y=utils.to_numpy(-log_probs), labels={"x": "position", "y": "loss (nats)"},
    title="Per-token loss on [random, same random]: the drop is in-context learning",
).add_vline(x=seq_len - 0.5, line_dash="dash").show()

Loss on first half : 14.92 nats
Loss on second half: 6.33 nats   <- should be dramatically lower


Now look at layer 1 on this input. Find heads 4 and 10 and look for the **offset diagonal stripe**.

In [15]:
for layer in range(model.cfg.n_layers):
    print(f"Layer {layer}")
    display(cv.attention.attention_patterns(tokens=rep_str, attention=rep_cache["pattern", layer]))

Layer 0


Layer 1


### ✏️ Code-along 3.2: Induction detector

Reuse `detect_heads` from 2.1. Just write the score function: mean attention on the diagonal at offset `-(seq_len - 1)`.

Hint: infer `seq_len` from the pattern shape, since the full length is `1 + 2 * seq_len`.

In [16]:
def induction_score(pattern: Float[Tensor, "seq_q seq_k"]) -> float:
    n = (pattern.shape[-1] - 1) // 2
    return pattern.diagonal(-(n - 1)).mean().item()

In [17]:
# ✅ check
found = detect_heads(rep_cache, induction_score)
print("Induction heads:", found)
assert set(found) == {"1.4", "1.10"}, "Expected exactly heads 1.4 and 1.10"

Induction heads: ['1.4', '1.10']


---
## 4️⃣ Hooks: measure at scale, then intervene

`run_with_cache` stores *everything* for *one* forward pass. That doesn't scale: a 12-layer model on a batch of 100 prompts means gigabytes of activations you mostly don't need. **Hooks** let you run *your function* on exactly the activations you care about, as the forward pass happens.

A hook function has this signature:

```python
def hook_fn(activation: Tensor, hook: HookPoint) -> Tensor | None:
    # read it, store something, or modify it
    # return None  -> activation unchanged (unless you edited it in place)
    # return x     -> x replaces the activation for the rest of the forward pass
```

and you attach it for a single run with:

```python
model.run_with_hooks(tokens, fwd_hooks=[(name_or_filter, hook_fn), ...], return_type=None)
```

- `name_or_filter` can be an exact name (`"blocks.1.attn.hook_pattern"`) or a function `name -> bool` to hit many hook points at once.
- `hook.layer()` tells you which layer you're in.
- Hooks are removed automatically after `run_with_hooks` returns.

💡 Hooks are the one primitive behind **every** technique in the rest of this notebook: reading (Part 4), attributing (Part 5), and ablating/patching (Part 6).

### ✏️ Code-along 4.1: Batched induction scores with a hook

Write `get_induction_scores(model, seq_len, batch_size)`, which returns a `[n_layers, n_heads]` tensor of induction scores averaged over a batch. Attach a hook to **every** `hook_pattern` via a name filter. In the hook, take the offset diagonal and average it over batch and position.

Hint: `pattern.diagonal(offset=..., dim1=-2, dim2=-1)` has shape `[batch, head, positions]`.

In [18]:
def get_induction_scores(model: HookedTransformer, seq_len: int = 50, batch_size: int = 10) -> Float[Tensor, "n_layers n_heads"]:
    tokens = generate_repeated_tokens(model, seq_len, batch_size)
    scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    def induction_score_hook(pattern: Float[Tensor, "batch head seq_q seq_k"], hook: HookPoint):
        stripe = pattern.diagonal(offset=1 - seq_len, dim1=-2, dim2=-1)  # [batch, head, positions]
        scores[hook.layer()] = einops.reduce(stripe, "batch head pos -> head", "mean")

    model.run_with_hooks(
        tokens,
        return_type=None,  # we only want the side effect, so skip computing logits
        fwd_hooks=[(lambda name: name.endswith("pattern"), induction_score_hook)],
    )
    return scores

In [19]:
# ✅ check
toy_induction_scores = get_induction_scores(model)
heatmap(toy_induction_scores, "Induction score by head (toy model)", text_auto=".2f")
assert toy_induction_scores[1, 4] > 0.5 and toy_induction_scores[1, 10] > 0.5

### ✏️ Code-along 4.2: Induction heads in a *real* model

Because you wrote a function of `model`, finding induction heads in GPT-2 is one line. This is the payoff of the tools-over-one-off-scripts mindset.

💡 **Why this matters:** Olsson et al. (2022) found induction heads in essentially every transformer they looked at, and their formation coincides with a visible "bump" in the training loss where in-context learning switches on. The toy circuit you're studying is a real, universal mechanism.

In [20]:
gpt2_induction_scores = get_induction_scores(gpt2_small)
heatmap(gpt2_induction_scores, "Induction score by head (GPT-2 small)", text_auto=".1f", height=550)

In [21]:
# ✅ check: the strongest heads should be in the middle layers (5-7)
top = t.topk(gpt2_induction_scores.flatten(), 5).indices
print("Top-5 induction heads in GPT-2:", [f"{i // 12}.{i % 12}" for i in top.tolist()])

Top-5 induction heads in GPT-2: ['5.5', '5.1', '6.9', '7.10', '7.2']


---
## 5️⃣ Who wrote the answer? Direct logit attribution

Back to the mental model. In our toy model (no LayerNorm, shortformer positions):

$$
\text{resid}_{\text{final}} \;=\; \underbrace{\text{embed}}_{\text{direct path}} \;+\; \sum_{h}\text{head}_{0.h} \;+\; \sum_{h}\text{head}_{1.h}
$$

(No positional embedding term: with shortformer, `pos_embed` never enters the residual stream.)

And `logits = resid_final @ W_U` is linear, so the logit for the **correct next token** `c` splits exactly:

$$
\text{logit}_c = \text{embed}\cdot W_U[:,c] \;+\; \sum_h \text{head}_{0.h}\cdot W_U[:,c] \;+\; \sum_h \text{head}_{1.h}\cdot W_U[:,c]
$$

Each term is that component's **direct contribution** to predicting the right answer.

💡 **Two caveats you should keep in mind (they motivate Part 6):**
1. It's *direct* only. Head 0.7 might barely write the answer itself but still be essential because head 1.4 *reads* it. DLA can't see that.
2. In real models the final LayerNorm is nonlinear. People still do DLA by "freezing" the LN scale, but then it's an approximation.

### ✏️ Code-along 5.1: Implement logit attribution

Return shape `[seq-1, 1 + 2*n_heads]`: column 0 is the direct path, then 12 columns for layer-0 heads, then 12 for layer 1.

In [22]:
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l0_results: Float[Tensor, "seq n_heads d_model"],
    l1_results: Float[Tensor, "seq n_heads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    W_U_correct = W_U[:, tokens[1:]]  # [d_model, seq-1]: direction for the right answer at each position

    direct = einops.einsum(W_U_correct, embed[:-1], "d seq, seq d -> seq")
    l0 = einops.einsum(W_U_correct, l0_results[:-1], "d seq, seq nh d -> seq nh")
    l1 = einops.einsum(W_U_correct, l1_results[:-1], "d seq, seq nh d -> seq nh")
    return t.cat([direct.unsqueeze(-1), l0, l1], dim=-1)

In [23]:
# ✅ check: the attributions must add up to the actual correct-token logits (up to the unembed bias)
tokens = model.to_tokens(text).squeeze()
logit_attr = logit_attribution(cache["embed"], cache["result", 0], cache["result", 1], model.W_U, tokens)

correct_logits = logits[0, t.arange(len(tokens) - 1), tokens[1:]]
t.testing.assert_close(logit_attr.sum(-1) + model.b_U[tokens[1:]], correct_logits, atol=1e-3, rtol=0)
print("Attributions sum exactly to the logits. The residual stream really is a sum.")

Attributions sum exactly to the logits. The residual stream really is a sum.


In [24]:
component_labels = ["direct"] + [f"0.{h}" for h in range(12)] + [f"1.{h}" for h in range(12)]


def plot_attr(attr, tokens, title):
    str_toks = model.to_str_tokens(tokens)
    fig = px.imshow(
        utils.to_numpy(attr.T), x=[f"{i}: {s!r}→{n!r}" for i, (s, n) in enumerate(zip(str_toks[:-1], str_toks[1:]))],
        y=component_labels, color_continuous_scale="RdBu", color_continuous_midpoint=0, title=title, height=650,
        labels={"x": "position (current→next)", "y": "component"}, aspect="auto",
    )
    fig.update_xaxes(showticklabels=False)
    fig.show()


plot_attr(logit_attr, tokens, "Logit attribution: normal text")

rep_attr = logit_attribution(rep_cache["embed"], rep_cache["result", 0], rep_cache["result", 1], model.W_U, rep_tokens.squeeze())
plot_attr(rep_attr, rep_tokens.squeeze(), "Logit attribution: repeated random tokens")

👀 **Read the second plot:** in the second half of the repeated sequence, the correct-token logit comes overwhelmingly from **1.4 and 1.10**. They *directly write the answer*, consistent with "induction heads copy B".

But notice **0.7 contributes almost nothing directly**. Is the previous-token head unimportant? DLA can't tell us. Time for causal interventions.

---
## 6️⃣ What breaks if I remove it? Ablation

Attribution tells you *who wrote* the answer. **Ablation** tells you *who's necessary*: knock a component out and measure how much worse the model does.

We ablate a head by zeroing its `z` (the pattern-weighted values, just before `W_O`), which deletes that head's write to the residual stream. Then measure loss on the **second half** of the repeated sequence (the part that needs induction).

💡 **Hook as intervention:** same hook machinery as Part 4, but now the hook **edits** the activation in place. Every downstream component sees the edited version, so we capture *indirect* effects too.

### ✏️ Code-along 6.1: Zero-ablation scan

Use `functools.partial` to bake `head_index_to_ablate` into the hook function.

In [25]:
def head_zero_ablation_hook(z: Float[Tensor, "batch seq n_heads d_head"], hook: HookPoint, head_index_to_ablate: int) -> None:
    z[:, :, head_index_to_ablate, :] = 0.0


def get_ablation_scores(
    model: HookedTransformer, tokens: Int[Tensor, "batch seq"], ablation_function: Callable = head_zero_ablation_hook
) -> Float[Tensor, "n_layers n_heads"]:
    '''[layer, head] = increase in loss on the second half when that head is ablated.'''
    n = (tokens.shape[1] - 1) // 2

    def second_half_loss(logits):
        return -get_log_probs(logits, tokens)[:, -(n - 1):].mean()

    clean_loss = second_half_loss(model(tokens))
    scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            hook_fn = functools.partial(ablation_function, head_index_to_ablate=head)
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)])
            scores[layer, head] = second_half_loss(ablated_logits) - clean_loss
    return scores

In [26]:
ablation_scores = get_ablation_scores(model, rep_tokens)
heatmap(ablation_scores, "Loss increase when each head is zero-ablated (repeated tokens)", text_auto=".2f")

# ✅ check: the circuit heads 0.7, 1.4 and 1.10 should all be among the most important
top4 = [f"{i // 12}.{i % 12}" for i in t.topk(ablation_scores.flatten(), 4).indices.tolist()]
print("Most important heads (descending):", top4)
print("Scores:", {h: round(ablation_scores[int(h[0]), int(h[2:])].item(), 2) for h in top4})
assert {"0.7", "1.4", "1.10"} <= set(top4)

Most important heads (descending): ['0.7', '1.10', '0.4', '1.4']
Scores: {'0.7': 8.07, '1.10': 5.43, '0.4': 2.96, '1.4': 1.77}


👀 **There it is.** Ablating **0.7** hurts the most of any head, even though its *direct* logit attribution was ~0. That's the signature of an **indirect effect**: 0.7's output is consumed by 1.4 and 1.10. We now have causal evidence for the full two-step story from Part 3.

Two things that *don't* fit the tidy story, which is normal in real interp work:
- **1.4 matters less than 1.10 under ablation**, even though both look like induction heads. The two partly back each other up: knock one out and the other still carries much of the load. Ablation measures *necessity*, and redundant components look less necessary than they are.
- **0.4 shows up too.** It's not part of the minimal circuit, and figuring out what it does is a (very hard) bonus in the ARENA notebook ("understand heads 0.4 & 0.11"). Keep a list of anomalies like this rather than explaining them away.

💡 **Zero vs. mean ablation.** Zeroing is a crude intervention, since a head's output might *never* be zero on real data. That pushes the model off-distribution. A gentler option is **mean ablation**: replace a head's output with its average over a batch. It preserves the "typical" output and removes only the *input-specific* information. (Swapping in activations from a *different* input is **activation patching**, the workhorse of circuit discovery in 1.4.1 IOI.)

### ✏️ Code-along 6.2 (short): Mean ablation

Replace the head's `z` with its mean over the batch dimension. Use a batch of 10 sequences.

In [27]:
def head_mean_ablation_hook(z: Float[Tensor, "batch seq n_heads d_head"], hook: HookPoint, head_index_to_ablate: int) -> None:
    z[:, :, head_index_to_ablate, :] = z[:, :, head_index_to_ablate, :].mean(0)

In [28]:
rep_tokens_batch = generate_repeated_tokens(model, seq_len=50, batch_size=10)
mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_function=head_mean_ablation_hook)
heatmap(mean_ablation_scores, "Loss increase under MEAN ablation", text_auto=".2f")

---
## 7️⃣ Read the algorithm straight from the weights

So far every result depended on the inputs we chose. The deepest claim in mech interp is that we can verify a circuit **from the weights alone**, for *every* possible token, without running the model.

### Recap: QK and OV as the two things a head does

For one head (row-vector convention, `x @ W`):

- **QK circuit** `W_QK = W_Q W_Kᵀ` (`d_model × d_model`): attention score between a query residual `x_q` and key residual `x_k` is `x_q W_QK x_kᵀ`.
- **OV circuit** `W_OV = W_V W_O` (`d_model × d_model`): if the head attends to `x_k`, it writes `x_k W_OV` into the stream.

Put the embedding in front and the unembedding behind, and you get **full circuits** that live in *token space*:

| Full circuit | Shape | Entry `[A, B]` means |
|---|---|---|
| `W_E W_OV W_U` (OV) | vocab × vocab | "If the head attends to token **A**, how much does it boost the logit of token **B**?" |
| `W_E W_QK W_Eᵀ` (QK) | vocab × vocab | "How much does query token **A** want to attend to key token **B**?" |

💡 **Prediction for a copying head:** its full OV circuit should look like the **identity**. Attending to A should boost the logit for A.

### The `FactoredMatrix` trick

`W_E W_V W_O W_U` is 50k × 50k = 2.5B floats ≈ **10 GB**. But it has **rank ≤ d_head = 64**, since it's a product through a 64-dim bottleneck. `FactoredMatrix(A, B)` stores `A` and `B` separately and does products, indexing, norms, eigenvalues, etc. *without materialising* `A @ B`. Use `.AB` to materialise a slice once you've indexed it down.

### ✏️ Code-along 7.1: The full OV circuit of head 1.4 is (mostly) a copier

1. Build `full_OV_circuit = W_E @ FactoredMatrix(W_V, W_O) @ W_U` for layer 1, head 4.
2. Write `top_1_acc`, the fraction of rows whose argmax is on the diagonal ("attending to A boosts A the most"). Process rows in batches, and materialise each slice with `.AB`.

In [29]:
layer, head = 1, 4
full_OV_circuit: FactoredMatrix = model.W_E @ FactoredMatrix(model.W_V[layer, head], model.W_O[layer, head]) @ model.W_U


def top_1_acc(full_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    '''Fraction of rows i where argmax_j full_circuit[i, j] == i.'''
    correct = 0
    for rows in t.split(t.arange(full_circuit.shape[0], device=device), batch_size):
        correct += (full_circuit[rows].AB.argmax(dim=-1) == rows).sum().item()
    return correct / full_circuit.shape[0]

In [30]:
# ✅ check + look at it: a random 200x200 sub-block should show a bright diagonal
assert full_OV_circuit.shape == (model.cfg.d_vocab, model.cfg.d_vocab)
idx = t.randint(0, model.cfg.d_vocab, (200,), device=device)
heatmap(full_OV_circuit[idx, idx].AB, "Full OV circuit of 1.4 (random 200 tokens)", "logit on output token", "attended-to token", height=600)

print(f"Head 1.4 alone: argmax on diagonal {top_1_acc(full_OV_circuit):.1%} of the time")

Head 1.4 alone: argmax on diagonal 30.8% of the time


The diagonal is clearly visible, but 1.4 alone isn't a perfect copier. Remember there are **two** induction heads. They could be splitting the job, each covering part of the vocabulary.

### ✏️ Code-along 7.2: Two heads, one copying circuit

If 1.4 and 1.10 attend to the same place (they do: same induction pattern), their outputs just add up. So their *effective* OV circuit is `W_V^{1.4} W_O^{1.4} + W_V^{1.10} W_O^{1.10}`.

That sum is itself a product of stacked matrices: concatenate the two `W_V`s along `d_head` (→ `d_model × 128`) and the two `W_O`s along `d_head` (→ `128 × d_model`). Build it as a `FactoredMatrix` and score it.

In [31]:
W_V_both = einops.rearrange(model.W_V[1, [4, 10]], "head d_model d_head -> d_model (head d_head)")
W_O_both = einops.rearrange(model.W_O[1, [4, 10]], "head d_head d_model -> (head d_head) d_model")
effective_OV = model.W_E @ FactoredMatrix(W_V_both, W_O_both) @ model.W_U

In [32]:
# ✅ check
acc = top_1_acc(effective_OV)
print(f"1.4 + 1.10 together: argmax on diagonal {acc:.1%} of the time")
assert acc > top_1_acc(full_OV_circuit)

1.4 + 1.10 together: argmax on diagonal 95.6% of the time


### ✏️ Code-along 7.3: Head 0.7 is a *positional* previous-token head

How does 0.7 know to attend to the previous token for **any** token? Hypothesis: it ignores token identity and uses **position only**.

With shortformer embeddings, the positional part of the QK circuit is `W_pos W_QK W_posᵀ` (n_ctx × n_ctx): "how much does query **position** i want to attend to key **position** j?". Compute it for head 0.7, apply the causal mask, scale by `1/√d_head`, softmax, and check that it's concentrated on the sub-diagonal.

In [33]:
layer, head = 0, 7
W_QK = model.W_Q[layer, head] @ model.W_K[layer, head].T
pos_scores = model.W_pos @ W_QK @ model.W_pos.T                        # [n_ctx, n_ctx]
causal = t.tril(t.ones_like(pos_scores, dtype=t.bool))
pos_by_pos_pattern = t.where(causal, pos_scores / model.cfg.d_head**0.5, -1e6).softmax(dim=-1)

In [34]:
# ✅ check
avg = pos_by_pos_pattern.diagonal(-1).mean().item()
print(f"Avg attention on previous position, over all 2048 positions: {avg:.3f}")
assert avg > 0.9
heatmap(pos_by_pos_pattern[:100, :100], "Positional QK pattern of 0.7 (first 100 positions)", "key position", "query position", height=550)

Avg attention on previous position, over all 2048 positions: 0.998


### ✏️ Code-along 7.4: The K-composition circuit, closing the loop

The last missing piece: **how does the induction head's query find the right key?** From Part 3:

- The **query** of 1.4 at position i is built from the *current token* A → `x_q = W_E[A]`.
- The **key** of 1.4 at position j is built from *what 0.7 wrote there*, namely 0.7's OV circuit applied to the *previous* token A' → `x_k = W_E[A'] W_OV^{0.7}`.

So the full K-composition QK circuit, in token space, is:

$$
\underbrace{W_E\, W_Q^{1.4}}_{\text{query side: current token}} \;\Big(\underbrace{W_E\, W_V^{0.7} W_O^{0.7}\, W_K^{1.4}}_{\text{key side: previous token via 0.7}}\Big)^{\!\top}
$$

Entry `[A, A']` = "how much does a query with current token A attend to a key whose previous token was A'?". For induction this should be **largest when A' = A**, i.e. the **identity** again!

Return it as a `FactoredMatrix(Q_side, K_side.T)`.

In [35]:
def find_K_comp_full_circuit(model: HookedTransformer, prev_token_head_index: int, ind_head_index: int) -> FactoredMatrix:
    '''[vocab, vocab]: rows = query-side current token, cols = key-side previous token (via the prev-token head).'''
    W_E = model.W_E
    W_Q, W_K = model.W_Q[1, ind_head_index], model.W_K[1, ind_head_index]
    W_V, W_O = model.W_V[0, prev_token_head_index], model.W_O[0, prev_token_head_index]

    Q_side = W_E @ W_Q                   # [vocab, d_head]
    K_side = W_E @ W_V @ W_O @ W_K       # [vocab, d_head]
    return FactoredMatrix(Q_side, K_side.T)

In [36]:
# ✅ check
K_comp_circuit = find_K_comp_full_circuit(model, prev_token_head_index=7, ind_head_index=4)
assert K_comp_circuit.shape == (model.cfg.d_vocab, model.cfg.d_vocab)
acc = top_1_acc(K_comp_circuit.T)
chance = 1 / model.cfg.d_vocab
print(f"0.7 -> 1.4 : for each previous-token A', the best-matching query token is A' itself {acc:.1%} of the time")
print(f"0.7 -> 1.10: {top_1_acc(find_K_comp_full_circuit(model, 7, 10).T):.1%}")
print(f"(chance would be {chance:.4%})")
assert acc > 1000 * chance

0.7 -> 1.4 : for each previous-token A', the best-matching query token is A' itself 52.0% of the time


0.7 -> 1.10: 55.2%
(chance would be 0.0020%)


**That's the whole circuit, verified from weights, for every token in the vocabulary:**

| Step | Head | Circuit | Weight-level evidence |
|---|---|---|---|
| "remember the previous token" | 0.7 | positional QK attends to i-1; OV passes token identity forward | 7.3: >90% on the sub-diagonal |
| "find where my token appeared before" | 1.4, 1.10 | **K-composition** with 0.7: query = current token, key = previous token via 0.7 | 7.4: identity-like, top-1 on the diagonal for a large fraction of a 50k vocab (chance ≈ 0.002%) |
| "copy what came next" | 1.4, 1.10 | OV circuit ≈ identity | 7.1 / 7.2: diagonal dominates |

And it agrees with the behavioural evidence (Part 3), the attributional evidence (Part 5), and the causal evidence (Part 6). **Four independent lines of evidence, one algorithm.** That convergence is what "understanding a circuit" means in mech interp.

### ✏️ Code-along 7.5 (bonus): Composition scores, finding the circuit *without* knowing it

In 7.4 we already knew which heads to connect. Can the weights tell us which layer-0 heads feed which layer-1 heads?

**Composition score** between a layer-0 OV matrix `W_A` and a layer-1 matrix `W_B`:

$$
\text{score}(W_A, W_B) = \frac{\lVert W_A W_B \rVert_F}{\lVert W_A \rVert_F \, \lVert W_B \rVert_F}
$$

Intuition: it's high when what `W_A` **writes** lies in the subspace `W_B` **reads**. Which `W_B` you pick determines the composition type:
- **Q-comp:** `W_B = W_QK^{1.j}` (layer-0 output feeds the query)
- **K-comp:** `W_B = (W_QK^{1.j})ᵀ` (feeds the key, as in induction)
- **V-comp:** `W_B = W_OV^{1.j}` (feeds the value)

In [37]:
def get_comp_score(W_A: Float[Tensor, "in_A out_A"], W_B: Float[Tensor, "out_A out_B"]) -> float:
    return ((W_A @ W_B).norm() / (W_A.norm() * W_B.norm())).item()

In [38]:
W_QK_all = model.W_Q @ model.W_K.transpose(-1, -2)   # [layer, head, d_model, d_model]
W_OV_all = model.W_V @ model.W_O

comp = {k: t.zeros(12, 12) for k in "QKV"}
for i in range(12):
    for j in range(12):
        comp["Q"][i, j] = get_comp_score(W_OV_all[0, i], W_QK_all[1, j])
        comp["K"][i, j] = get_comp_score(W_OV_all[0, i], W_QK_all[1, j].T)
        comp["V"][i, j] = get_comp_score(W_OV_all[0, i], W_OV_all[1, j])

for k in "QKV":
    heatmap(comp[k], f"{k}-composition scores", "layer-1 head", "layer-0 head", text_auto=".2f", zmin=0)

# ✅ check: the strongest K-composition edges into 1.4 / 1.10 come from 0.7
print("Strongest K-comp source into 1.4:", f"0.{comp['K'][:, 4].argmax().item()}")
print("Strongest K-comp source into 1.10:", f"0.{comp['K'][:, 10].argmax().item()}")

Strongest K-comp source into 1.4: 0.7
Strongest K-comp source into 1.10: 0.7


Look at the K-composition heatmap: the brightest cells are **0.7 → 1.4** and **0.7 → 1.10**. The weights alone point you at the circuit.

⚠️ **Calibrate before you trust a number.** Even random matrices get a non-zero composition score (roughly `1/√d_head` scale). Always compare against a baseline. The ARENA notebook's "Setting a Baseline" exercise does this properly with random matrices of the same shapes.

---
## 🧭 What you now have

**Concepts**
- The residual stream is a sum of component outputs ⇒ linear decompositions (DLA, decomposed QK inputs) are *exact* in attention-only models without LN.
- A head is a **QK** circuit (where to look) plus an **OV** circuit (what to move). Full circuits put these in token space.
- Multi-layer algorithms happen through **composition**; induction is K-composition of a previous-token head into a copying head.
- Evidence types: **patterns** (what it looks at) → **attribution** (who writes the answer) → **ablation** (who is necessary) → **weights** (why, for all inputs).

**Tools (all functions of `model`, so they transfer to other models)**
`detect_heads` · `get_induction_scores` · `logit_attribution` · `get_ablation_scores` · `top_1_acc` · `find_K_comp_full_circuit` · `get_comp_score`

**Where to go next**
- **ARENA 1.2 (full)**: bonus sections, decomposing attention scores by component, baselining composition scores, batched `FactoredMatrix` composition.
- **ARENA 1.4.1 IOI**: the same toolkit (plus activation patching and path patching) applied to a 26-head circuit in GPT-2.
- **Paper:** *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021), where every idea in this notebook comes from. Read it now; it'll feel familiar.